# Meta-Learner Feature Engineering And XGBoost Classifier


## Configuration


## Imports


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_validate
from xgboost import XGBClassifier

from meta_learner_classifier_tsplit_utils import (
    TARGET_CLASS_COL,
    TemporalDecaySampleWeightClassifier,
    build_classifier_scoring,
    build_meta_classifier_dataset_from_feature_frame,
    evaluate_baseline_hard_class_over_splits,
    fit_best_xgb_classifier,
    print_classifier_cv_metrics,
    select_best_trial_min_log_loss_classifier,
    summarize_classifier_lexicographic_candidates,
    summarize_classifier_optuna_trials,
    summarize_classifier_predictions,
    summarize_confidence_thresholds,
    summarize_day_by_day_classifier_thresholds,
    summarize_hard_class_predictions,
    tune_xgb_classifier_optuna,
)
from meta_learner_feature_utils import (
    build_meta_learner_feature_frame,
    load_meta_learner_dataframe,
)
from nba_ou.modeling.modeling import (
    ModelBundleMetadata,
    ModelInfo,
    TrainingMetrics,
    assert_valid_time_splits,
    evaluate_day_by_day_walk_forward,
    make_test_anchored_walk_forward_splits,
    save_model_bundle,
    split_latest_dates_holdout,
)

pd.options.display.float_format = '{:,.4f}'.format


In [2]:
CSV_PATH = '/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/meta_learner_last_3_seasons_20260408.csv'
OUTPUT_FEATURE_CSV = None

ROLLING_WINDOW_DAYS = 3
LINE_BUCKET_EDGES = [205.0, 215.0, 225.0, 235.0, 245.0]
INCLUDE_BASELINE_FEATURES = False
FINAL_HOLDOUT_SIZE = 0.08

TRAIN_GAMES = 2500
TEST_GAMES = 30
STEP_GAMES_BETWEEN_TESTS = 40
MIN_TRAIN_GAMES = int(TRAIN_GAMES * 0.75)
MAX_FOLDS = 15

SAMPLE_WEIGHT_LAMBDA = 0.00075
SAMPLE_WEIGHT_LAMBDA_BOUNDS = (1e-4, 0.02)
CONFIDENCE_THRESHOLDS = (0.50, 0.52, 0.55, 0.58, 0.60, 0.65)
XGB_FIXED_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'tree_method': 'hist',
    'max_depth': 2,
    'learning_rate': 0.011,
    'n_estimators': 60,
    'subsample': 0.568,
    'colsample_bytree': 0.548,
    'reg_alpha': 1.877,
    'reg_lambda': 1.897,
    'min_child_weight': 5.387,
    'gamma': 1.652,
    'n_jobs': -1,
    'random_state': 16,
}

OPTUNA_N_TRIALS = 60
OPTUNA_TIMEOUT_SECONDS = int(1 * 3600)
OPTUNA_LOG_LOSS_TOLERANCE_ABS = 0.01
MODEL_OUT_DIR = '/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/meta_learner/classifier/'


## Load Data


In [3]:
raw_df = load_meta_learner_dataframe(CSV_PATH)

feature_result = build_meta_learner_feature_frame(
    raw_df,
    rolling_window_days=ROLLING_WINDOW_DAYS,
    line_bucket_edges=LINE_BUCKET_EDGES,
)

df_features = feature_result.dataframe.copy()

print(f'CSV path: {CSV_PATH}')
print(f'Rows in feature frame: {len(df_features):,}')
print(f'Columns in feature frame: {len(df_features.columns):,}')
print(f'Line column: {feature_result.line_col}')
print(f'Total-points model columns: {feature_result.total_model_cols}')
print(f'Line-error model columns: {feature_result.line_error_model_cols}')
print(f'Error-space prediction columns: {feature_result.error_cols}')
print(f'Vote feature count: {len(feature_result.vote_feature_cols)}')
print(f'Reliability feature count: {len(feature_result.reliability_feature_cols)}')


CSV path: /home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/meta_learner_last_3_seasons_20260408.csv
Rows in feature frame: 3,264
Columns in feature frame: 833
Line column: TOTAL_LINE_bet365
Total-points model columns: ['PRED_TOTAL_POINTS_FULL_DATASET', 'PRED_TOTAL_POINTS_LAST_5_SEASONS', 'PRED_TOTAL_POINTS_LAST_3_SEASONS']
Line-error model columns: ['PRED_LINE_ERROR_FULL_DATASET', 'PRED_LINE_ERROR_LAST_5_SEASONS', 'PRED_LINE_ERROR_LAST_3_SEASONS']
Error-space prediction columns: ['PRED_TOTAL_POINTS_FULL_DATASET__ERR', 'PRED_TOTAL_POINTS_LAST_5_SEASONS__ERR', 'PRED_TOTAL_POINTS_LAST_3_SEASONS__ERR', 'PRED_LINE_ERROR_FULL_DATASET__ERR', 'PRED_LINE_ERROR_LAST_5_SEASONS__ERR', 'PRED_LINE_ERROR_LAST_3_SEASONS__ERR']
Vote feature count: 66
Reliability feature count: 356


## Train / Test


In [4]:
dataset = build_meta_classifier_dataset_from_feature_frame(
    df_features,
    include_baseline_features=INCLUDE_BASELINE_FEATURES,
)

df_model = dataset.dataframe.reset_index(drop=True).copy()
X_all = dataset.feature_frame.reset_index(drop=True).copy()
y_all = dataset.target.reset_index(drop=True).copy()

print(f'Rows after removing pushes: {len(df_model):,}')
print('Season counts:')
print(df_model['SEASON_YEAR'].value_counts().sort_index())
print(f'Feature count kept: {len(dataset.feature_cols)}')
print(f'Baseline features included as model inputs: {INCLUDE_BASELINE_FEATURES}')
print(f'Dropped non-numeric feature columns: {len(dataset.dropped_feature_cols)}')
if dataset.dropped_feature_cols:
    print(dataset.dropped_feature_cols[:20])


Rows after removing pushes: 3,229
Season counts:
SEASON_YEAR
2023     915
2024    1219
2025    1095
Name: count, dtype: int64
Feature count kept: 822
Baseline features included as model inputs: False
Dropped non-numeric feature columns: 0


In [5]:
df_split = df_model.copy()
df_split['__row_id__'] = np.arange(len(df_split))

df_dev, df_test_final = split_latest_dates_holdout(
    df=df_split,
    date_col='GAME_DATE',
    test_size=FINAL_HOLDOUT_SIZE,
)

dev_idx = df_dev['__row_id__'].to_numpy()
test_idx = df_test_final['__row_id__'].to_numpy()

X_dev = X_all.iloc[dev_idx].copy()
X_test_final = X_all.iloc[test_idx].copy()
y_dev = y_all.iloc[dev_idx].copy()
y_test_final = y_all.iloc[test_idx].copy()

print(f'Development rows: {len(df_dev):,}')
print(f'Final holdout rows: {len(df_test_final):,}')
print('Final holdout date range:', df_test_final['GAME_DATE'].min(), '->', df_test_final['GAME_DATE'].max())
print('Development season counts:')
print(df_dev['SEASON_YEAR'].value_counts().sort_index())
print('Final holdout season counts:')
print(df_test_final['SEASON_YEAR'].value_counts().sort_index())


Development rows: 2,966
Final holdout rows: 263
Final holdout date range: 2026-03-04 00:00:00 -> 2026-04-08 00:00:00
Development season counts:
SEASON_YEAR
2023     915
2024    1219
2025     832
Name: count, dtype: int64
Final holdout season counts:
SEASON_YEAR
2025    263
Name: count, dtype: int64


In [6]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col='GAME_DATE',
    season_col='SEASON_YEAR',
    test_games=TEST_GAMES,
    step_games_between_tests=STEP_GAMES_BETWEEN_TESTS,
    train_games=TRAIN_GAMES,
    min_train_games=MIN_TRAIN_GAMES,
    max_folds=MAX_FOLDS,
    verbose=1,
)

assert_valid_time_splits(df_dev, splits, date_col='GAME_DATE')
display(fold_info)


Created 14 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           1883            35       2023-10-30     2025-03-21      2025-03-22    2025-03-26         2024
    2           1963            30       2023-10-30     2025-04-01      2025-04-02    2025-04-05         2024
    3           2043            31       2023-10-30     2025-04-11      2025-04-13    2025-04-25         2024
    4           2134            32       2023-10-30     2025-06-22      2025-10-27    2025-11-03         2025
    5           2210            30       2023-10-30     2025-11-09      2025-11-10    2025-11-13         2025
    6           2285            38       2023-10-30     2025-11-19      2025-11-20    2025-11-24         2025
    7           2371            32       2023-10-30     2025-12-01      2025-12-02    2025-12-05         2025
    8           2445            34       2023-10-30     2025-12-14      2025

,fold,train_n_games,test_n_games,train_start_date,train_end_date,test_start_date,test_end_date,test_season
0,1,1883,35,2023-10-30,2025-03-21,2025-03-22,2025-03-26,2024
1,2,1963,30,2023-10-30,2025-04-01,2025-04-02,2025-04-05,2024
2,3,2043,31,2023-10-30,2025-04-11,2025-04-13,2025-04-25,2024
3,4,2134,32,2023-10-30,2025-06-22,2025-10-27,2025-11-03,2025
4,5,2210,30,2023-10-30,2025-11-09,2025-11-10,2025-11-13,2025
5,6,2285,38,2023-10-30,2025-11-19,2025-11-20,2025-11-24,2025
6,7,2371,32,2023-10-30,2025-12-01,2025-12-02,2025-12-05,2025
7,8,2445,34,2023-10-30,2025-12-14,2025-12-15,2025-12-20,2025
8,9,2500,30,2023-11-05,2025-12-26,2025-12-27,2025-12-30,2025
9,10,2500,39,2023-11-15,2026-01-04,2026-01-05,2026-01-09,2025


## Baselines


In [7]:
baseline_cv_summary = evaluate_baseline_hard_class_over_splits(
    df=df_dev,
    target_col=TARGET_CLASS_COL,
    baseline_error_cols=dataset.baseline_cols,
    splits=splits,
)

baseline_final_summary = pd.DataFrame(
    [
        summarize_hard_class_predictions(
            y_true=y_test_final,
            y_pred=(pd.to_numeric(df_test_final[column], errors='coerce') > 0).astype(int),
            label=column,
        )
        for column in dataset.baseline_cols
    ]
)

print('Cross-validation baselines')
display(baseline_cv_summary.round(4))
print('Final holdout baselines')
display(baseline_final_summary.round(4))


Cross-validation baselines


,model,validation_accuracy_pct,validation_balanced_accuracy_pct
0,BASE_AVG_ALL_6_ERR,54.7393,52.2378
1,BASE_MAJORITY_TOTAL_ONLY_ERR,55.4670,53.7951
2,BASE_MAJORITY_LINE_ERROR_ONLY_ERR,53.6255,51.3590
3,BASE_MAJORITY_ALL_6_TIE_WITH_PRED_LINE_ERROR_F...,55.9866,53.2463


Final holdout baselines


,model,n_games,accuracy_pct,balanced_accuracy_pct
0,BASE_AVG_ALL_6_ERR,263,50.5703,50.8417
1,BASE_MAJORITY_TOTAL_ONLY_ERR,263,51.3308,51.6025
2,BASE_MAJORITY_LINE_ERROR_ONLY_ERR,263,53.2319,53.4103
3,BASE_MAJORITY_ALL_6_TIE_WITH_PRED_LINE_ERROR_F...,263,52.8517,53.0661


In [8]:
SCORING = build_classifier_scoring()
DAY_BY_DAY_METRIC_NAME = 'Accuracy'


def print_final_metrics(label, y_true, y_proba):
    summary = pd.DataFrame(
        [
            summarize_classifier_predictions(
                y_true=y_true,
                y_proba=y_proba,
                label=label,
            )
        ]
    )
    display(summary.round(4))
    return summary


def run_day_by_day_classifier_evaluation(
    *,
    label,
    df_dev,
    df_test_final,
    fit_and_predict,
    max_games=TRAIN_GAMES,
    thresholds=CONFIDENCE_THRESHOLDS,
):
    result = evaluate_day_by_day_walk_forward(
        df_dev=df_dev,
        df_test_final=df_test_final,
        fit_and_predict=fit_and_predict,
        metric_fn=lambda y_true, y_pred: accuracy_score(
            np.asarray(y_true, dtype=int),
            (np.asarray(y_pred, dtype=float) >= 0.5).astype(int),
        ),
        target_col=TARGET_CLASS_COL,
        date_col='GAME_DATE',
        max_games=max_games,
        metric_name=DAY_BY_DAY_METRIC_NAME,
    )

    threshold_results = summarize_day_by_day_classifier_thresholds(
        result.predictions,
        thresholds=thresholds,
    )

    print(f'{label} mean day-by-day {DAY_BY_DAY_METRIC_NAME}: {result.mean_metric:.2%}')
    display(result.daily_results.style.format({DAY_BY_DAY_METRIC_NAME: '{:.2%}'}))
    print(f'{label} thresholded walk-forward accuracy')
    display(
        threshold_results.style.format(
            {'pct_of_test': '{:.1%}', 'bet_accuracy_pct': '{:.2f}', 'mean_confidence_pct': '{:.2f}'}
        )
    )
    return result, threshold_results

## XGBoost No Sample Weights


In [9]:
xgb_clf_no_weights = XGBClassifier(**XGB_FIXED_PARAMS)

cv_results_no_weights = cross_validate(
    xgb_clf_no_weights,
    X_dev,
    y_dev,
    cv=splits,
    scoring=SCORING,
    return_train_score=True,
    n_jobs=1,
)

print('XGBoost classifier no sample weights')
print_classifier_cv_metrics(cv_results_no_weights, SCORING)

XGBoost classifier no sample weights
Train Accuracy: 61.70%
Validation Accuracy: 54.22%

Train Balanced_Accuracy: 60.74%
Validation Balanced_Accuracy: 51.03%

Train Brier: 0.24405
Validation Brier: 0.24847

Train LogLoss: 0.68123
Validation LogLoss: 0.69008



In [10]:
xgb_clf_no_weights.fit(X_dev, y_dev)
y_proba_test_no_weights = xgb_clf_no_weights.predict_proba(X_test_final)[:, 1]

print('Final holdout metrics')
final_summary_no_weights = print_final_metrics(
    'XGBoost classifier no sample weights',
    y_test_final,
    y_proba_test_no_weights,
)

Final holdout metrics


,model,n_games,accuracy_pct,balanced_accuracy_pct,brier_score,log_loss,mean_predicted_over_pct,mean_confidence_pct
0,XGBoost classifier no sample weights,263,50.1901,50.9459,0.2500,0.6931,48.6108,51.5143


In [11]:
threshold_summary_no_weights = summarize_confidence_thresholds(
    y_true=y_test_final,
    y_proba=y_proba_test_no_weights,
    thresholds=CONFIDENCE_THRESHOLDS,
)

display(
    threshold_summary_no_weights.style.format(
        {'pct_of_test': '{:.1%}', 'bet_accuracy_pct': '{:.2f}', 'mean_confidence_pct': '{:.2f}'}
    )
)

,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,0.500000,263,100.0%,50.19,51.51
1,0.520000,81,30.8%,50.62,52.54
2,0.550000,0,0.0%,nan,nan
3,0.580000,0,0.0%,nan,nan
4,0.600000,0,0.0%,nan,nan
5,0.650000,0,0.0%,nan,nan


In [12]:
def fit_and_predict_xgb_no_weights_day_by_day(train_df, test_df):
    model = XGBClassifier(**XGB_FIXED_PARAMS)

    train_rows = train_df['__row_id__'].to_numpy(dtype=int)
    test_rows = test_df['__row_id__'].to_numpy(dtype=int)

    X_train = X_all.iloc[train_rows]
    y_train = y_all.iloc[train_rows]
    X_test = X_all.iloc[test_rows]

    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]


day_by_day_no_weights, day_by_day_no_weights_thresholds = run_day_by_day_classifier_evaluation(
    label='XGBoost classifier no sample weights',
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_no_weights_day_by_day,
    max_games=TRAIN_GAMES,
)

XGBoost classifier no sample weights mean day-by-day Accuracy: 51.53%


,date,train_n_games,test_n_games,train_start_date,train_end_date,Accuracy
0,2026-03-04 00:00:00,2500,5,2024-02-13 00:00:00,2026-03-03 00:00:00,60.00%
1,2026-03-05 00:00:00,2500,9,2024-02-14 00:00:00,2026-03-04 00:00:00,77.78%
2,2026-03-06 00:00:00,2500,7,2024-02-14 00:00:00,2026-03-05 00:00:00,28.57%
3,2026-03-07 00:00:00,2500,5,2024-02-23 00:00:00,2026-03-06 00:00:00,80.00%
4,2026-03-08 00:00:00,2500,8,2024-02-24 00:00:00,2026-03-07 00:00:00,50.00%
5,2026-03-09 00:00:00,2500,5,2024-02-25 00:00:00,2026-03-08 00:00:00,40.00%
6,2026-03-10 00:00:00,2500,10,2024-02-26 00:00:00,2026-03-09 00:00:00,50.00%
7,2026-03-11 00:00:00,2500,6,2024-02-27 00:00:00,2026-03-10 00:00:00,50.00%
8,2026-03-12 00:00:00,2500,8,2024-02-28 00:00:00,2026-03-11 00:00:00,37.50%
9,2026-03-13 00:00:00,2500,8,2024-02-29 00:00:00,2026-03-12 00:00:00,50.00%


XGBoost classifier no sample weights thresholded walk-forward accuracy


,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,0.500000,263,100.0%,52.09,51.64
1,0.520000,87,33.1%,49.43,52.86
2,0.550000,2,0.8%,100.00,55.00
3,0.580000,0,0.0%,nan,nan
4,0.600000,0,0.0%,nan,nan
5,0.650000,0,0.0%,nan,nan


## Check Weighted


In [13]:
xgb_clf_weighted = XGBClassifier(**XGB_FIXED_PARAMS)
weighted_xgb_clf = TemporalDecaySampleWeightClassifier(
    estimator=xgb_clf_weighted,
    dates=df_dev['GAME_DATE'],
    lambda_=SAMPLE_WEIGHT_LAMBDA,
)

cv_results_weights = cross_validate(
    weighted_xgb_clf,
    X_dev,
    y_dev,
    cv=splits,
    scoring=SCORING,
    return_train_score=True,
    n_jobs=1,
)

print('XGBoost classifier with sample weights')
print_classifier_cv_metrics(cv_results_weights, SCORING)

XGBoost classifier with sample weights
Train Accuracy: 62.08%
Validation Accuracy: 55.37%

Train Balanced_Accuracy: 61.15%
Validation Balanced_Accuracy: 53.01%

Train Brier: 0.24426
Validation Brier: 0.24859

Train LogLoss: 0.68165
Validation LogLoss: 0.69032



In [14]:
weighted_xgb_clf.fit(X_dev, y_dev)
y_proba_test_weights = weighted_xgb_clf.predict_proba(X_test_final)[:, 1]

print('Final holdout metrics')
final_summary_weights = print_final_metrics(
    'XGBoost classifier with sample weights',
    y_test_final,
    y_proba_test_weights,
)

Final holdout metrics


,model,n_games,accuracy_pct,balanced_accuracy_pct,brier_score,log_loss,mean_predicted_over_pct,mean_confidence_pct
0,XGBoost classifier with sample weights,263,48.6692,49.5401,0.2505,0.6942,47.8655,52.1825


In [15]:
threshold_summary_weights = summarize_confidence_thresholds(
    y_true=y_test_final,
    y_proba=y_proba_test_weights,
    thresholds=CONFIDENCE_THRESHOLDS,
)

display(
    threshold_summary_weights.style.format(
        {'pct_of_test': '{:.1%}', 'bet_accuracy_pct': '{:.2f}', 'mean_confidence_pct': '{:.2f}'}
    )
)

,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,0.500000,263,100.0%,48.67,52.18
1,0.520000,158,60.1%,52.53,52.88
2,0.550000,0,0.0%,nan,nan
3,0.580000,0,0.0%,nan,nan
4,0.600000,0,0.0%,nan,nan
5,0.650000,0,0.0%,nan,nan


In [16]:
def fit_and_predict_xgb_weights_day_by_day(train_df, test_df):
    base_model = XGBClassifier(**XGB_FIXED_PARAMS)

    train_rows = train_df['__row_id__'].to_numpy(dtype=int)
    test_rows = test_df['__row_id__'].to_numpy(dtype=int)

    X_train = X_all.iloc[train_rows]
    y_train = y_all.iloc[train_rows]
    X_test = X_all.iloc[test_rows]

    model = TemporalDecaySampleWeightClassifier(
        estimator=base_model,
        dates=train_df['GAME_DATE'],
        lambda_=SAMPLE_WEIGHT_LAMBDA,
    )

    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]


day_by_day_weights, day_by_day_weights_thresholds = run_day_by_day_classifier_evaluation(
    label='XGBoost classifier with sample weights',
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_weights_day_by_day,
    max_games=TRAIN_GAMES,
)

XGBoost classifier with sample weights mean day-by-day Accuracy: 50.54%


,date,train_n_games,test_n_games,train_start_date,train_end_date,Accuracy
0,2026-03-04 00:00:00,2500,5,2024-02-13 00:00:00,2026-03-03 00:00:00,60.00%
1,2026-03-05 00:00:00,2500,9,2024-02-14 00:00:00,2026-03-04 00:00:00,77.78%
2,2026-03-06 00:00:00,2500,7,2024-02-14 00:00:00,2026-03-05 00:00:00,28.57%
3,2026-03-07 00:00:00,2500,5,2024-02-23 00:00:00,2026-03-06 00:00:00,80.00%
4,2026-03-08 00:00:00,2500,8,2024-02-24 00:00:00,2026-03-07 00:00:00,50.00%
5,2026-03-09 00:00:00,2500,5,2024-02-25 00:00:00,2026-03-08 00:00:00,40.00%
6,2026-03-10 00:00:00,2500,10,2024-02-26 00:00:00,2026-03-09 00:00:00,50.00%
7,2026-03-11 00:00:00,2500,6,2024-02-27 00:00:00,2026-03-10 00:00:00,50.00%
8,2026-03-12 00:00:00,2500,8,2024-02-28 00:00:00,2026-03-11 00:00:00,37.50%
9,2026-03-13 00:00:00,2500,8,2024-02-29 00:00:00,2026-03-12 00:00:00,50.00%


XGBoost classifier with sample weights thresholded walk-forward accuracy


,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,0.500000,263,100.0%,51.33,51.79
1,0.520000,110,41.8%,51.82,52.80
2,0.550000,2,0.8%,100.00,55.13
3,0.580000,0,0.0%,nan,nan
4,0.600000,0,0.0%,nan,nan
5,0.650000,0,0.0%,nan,nan


In [17]:
cv_classifier_summary = pd.DataFrame(
    [
        {
            'model': 'XGBoost classifier no sample weights',
            'validation_accuracy_pct': 100.0 * cv_results_no_weights['test_Accuracy'].mean(),
            'validation_balanced_accuracy_pct': 100.0 * cv_results_no_weights['test_Balanced_Accuracy'].mean(),
            'validation_brier_score': -cv_results_no_weights['test_Brier'].mean(),
            'validation_log_loss': -cv_results_no_weights['test_LogLoss'].mean(),
        },
        {
            'model': 'XGBoost classifier with sample weights',
            'validation_accuracy_pct': 100.0 * cv_results_weights['test_Accuracy'].mean(),
            'validation_balanced_accuracy_pct': 100.0 * cv_results_weights['test_Balanced_Accuracy'].mean(),
            'validation_brier_score': -cv_results_weights['test_Brier'].mean(),
            'validation_log_loss': -cv_results_weights['test_LogLoss'].mean(),
        },
    ]
)

final_classifier_summary = pd.concat(
    [final_summary_no_weights, final_summary_weights],
    ignore_index=True,
)

print('Cross-validation comparison')
display(
    pd.concat(
        [baseline_cv_summary, cv_classifier_summary],
        ignore_index=True,
    ).round(4)
)

print('Final holdout comparison')
display(
    pd.concat(
        [
            baseline_final_summary,
            final_classifier_summary,
        ],
        ignore_index=True,
    ).round(4)
)


Cross-validation comparison


,model,validation_accuracy_pct,validation_balanced_accuracy_pct,validation_brier_score,validation_log_loss
0,BASE_AVG_ALL_6_ERR,54.7393,52.2378,NaN,NaN
1,BASE_MAJORITY_TOTAL_ONLY_ERR,55.4670,53.7951,NaN,NaN
2,BASE_MAJORITY_LINE_ERROR_ONLY_ERR,53.6255,51.3590,NaN,NaN
3,BASE_MAJORITY_ALL_6_TIE_WITH_PRED_LINE_ERROR_F...,55.9866,53.2463,NaN,NaN
4,XGBoost classifier no sample weights,54.2199,51.0283,0.2485,0.6901
5,XGBoost classifier with sample weights,55.3726,53.0129,0.2486,0.6903


Final holdout comparison


,model,n_games,accuracy_pct,balanced_accuracy_pct,brier_score,log_loss,mean_predicted_over_pct,mean_confidence_pct
0,BASE_AVG_ALL_6_ERR,263,50.5703,50.8417,NaN,NaN,NaN,NaN
1,BASE_MAJORITY_TOTAL_ONLY_ERR,263,51.3308,51.6025,NaN,NaN,NaN,NaN
2,BASE_MAJORITY_LINE_ERROR_ONLY_ERR,263,53.2319,53.4103,NaN,NaN,NaN,NaN
3,BASE_MAJORITY_ALL_6_TIE_WITH_PRED_LINE_ERROR_F...,263,52.8517,53.0661,NaN,NaN,NaN,NaN
4,XGBoost classifier no sample weights,263,50.1901,50.9459,0.2500,0.6931,48.6108,51.5143
5,XGBoost classifier with sample weights,263,48.6692,49.5401,0.2505,0.6942,47.8655,52.1825


# Optuna


In [18]:
study = tune_xgb_classifier_optuna(
    X=X_dev,
    y=y_dev,
    sample_weight_dates=df_dev['GAME_DATE'],
    tune_sample_weight_lambda=True,
    sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    splits=splits,
    n_trials=OPTUNA_N_TRIALS,
    timeout=OPTUNA_TIMEOUT_SECONDS,
    study_name='meta_learner_xgb_classifier_logloss',
)

best_trial_min_loss = select_best_trial_min_log_loss_classifier(study)

print('Optuna best by log loss only')
print('Trial:', study.best_trial.number)
print('Best CV log loss:', study.best_value)
print('Mean accuracy:', study.best_trial.user_attrs.get('mean_accuracy'))
print('Mean balanced accuracy:', study.best_trial.user_attrs.get('mean_balanced_accuracy'))
print('Mean Brier:', study.best_trial.user_attrs.get('mean_brier'))

print('\nSelected trial for fitting: minimum CV log loss')
print('Trial:', best_trial_min_loss.number)
print('CV log loss:', best_trial_min_loss.user_attrs.get('mean_log_loss', best_trial_min_loss.value))
print('Mean accuracy:', best_trial_min_loss.user_attrs.get('mean_accuracy'))
print('Mean balanced accuracy:', best_trial_min_loss.user_attrs.get('mean_balanced_accuracy'))
print('Mean Brier:', best_trial_min_loss.user_attrs.get('mean_brier'))
print('Median best_iteration:', best_trial_min_loss.user_attrs.get('median_best_iteration'))
print('Params:')
for k, v in best_trial_min_loss.params.items():
    print(f'{k}: {v}')

trials_df = summarize_classifier_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            'value_log_loss': '{:.4f}',
            'mean_log_loss': '{:.4f}',
            'mean_accuracy': '{:.2%}',
            'mean_balanced_accuracy': '{:.2%}',
            'mean_brier': '{:.4f}',
        }
    )
)

candidates_df = summarize_classifier_lexicographic_candidates(
    study,
    log_loss_tolerance_abs=OPTUNA_LOG_LOSS_TOLERANCE_ABS,
)
display(
    candidates_df.head(15).style.format(
        {
            'value_log_loss': '{:.4f}',
            'mean_log_loss': '{:.4f}',
            'mean_accuracy': '{:.2%}',
            'mean_balanced_accuracy': '{:.2%}',
            'mean_brier': '{:.4f}',
        }
    )
)


[I 2026-04-10 13:32:58,630] A new study created in memory with name: meta_learner_xgb_classifier_logloss


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-04-10 13:35:25,786] Trial 0 finished with value: 0.6837648658516639 and parameters: {'max_depth': 2, 'min_child_weight': 5.387049498103234, 'gamma': 1.6521043697435434, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.5484008594412749, 'learning_rate': 0.01145142679635112, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 1.897469395521307, 'sample_weight_lambda': 0.0001451508648952848}. Best is trial 0 with value: 0.6837648658516639.
[I 2026-04-10 13:40:46,743] Trial 1 finished with value: 0.6748885431468047 and parameters: {'max_depth': 4, 'min_child_weight': 6.137517643162606, 'gamma': 0.2339770181699612, 'subsample': 0.8390562044499359, 'colsample_bytree': 0.43714869543774154, 'learning_rate': 0.012057859517190121, 'reg_alpha': 0.09307011182812809, 'reg_lambda': 15.258811505244246, 'sample_weight_lambda': 0.0011702687079247593}. Best is trial 1 with value: 0.6748885431468047.
[I 2026-04-10 13:42:52,474] Trial 2 finished with value: 0.6740535716150904 and parameters: {'ma

,trial,value_log_loss,mean_accuracy,mean_balanced_accuracy,mean_brier,mean_log_loss,mean_best_iteration,median_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,17,0.6605,59.05%,58.15%,0.2343,0.6605,62,12,4,2.260454,1.857087,0.727676,0.772640,0.048492,0.512588,1.064875,0.007335
1,22,0.6655,59.62%,58.68%,0.2366,0.6655,64,25,3,1.416640,2.625159,0.720908,0.554465,0.049555,0.666938,1.032890,0.008786
2,21,0.6675,59.29%,59.20%,0.2376,0.6675,54,14,3,1.031972,2.591255,0.732149,0.603324,0.047888,1.030835,1.010468,0.007651
3,19,0.6675,57.08%,56.49%,0.2375,0.6675,47,17,3,1.022417,2.665636,0.746679,0.558282,0.046173,0.846720,1.035041,0.007180
4,18,0.6684,59.05%,58.32%,0.2379,0.6684,42,10,4,1.070178,2.475553,0.748674,0.592791,0.049794,1.554391,1.043063,0.006913
5,9,0.6708,57.03%,56.63%,0.2390,0.6708,47,20,4,4.333635,1.201269,0.669576,0.805598,0.042139,3.380961,1.306366,0.002480
6,13,0.6735,61.62%,59.71%,0.2403,0.6735,88,43,4,14.829327,1.321746,0.687146,0.893901,0.025634,0.355407,3.874519,0.000129
7,6,0.6739,62.52%,57.94%,0.2405,0.6739,79,21,4,6.557627,1.260311,0.833701,0.884625,0.034871,0.264001,42.379142,0.000129
8,2,0.6741,61.71%,59.76%,0.2406,0.6741,94,44,2,4.213435,0.362636,0.917075,0.661545,0.018364,0.109243,1.184192,0.005388
9,1,0.6749,60.81%,57.20%,0.2410,0.6749,160,45,4,6.137518,0.233977,0.839056,0.437149,0.012058,0.093070,15.258812,0.001170


,trial,value_log_loss,mean_log_loss,mean_accuracy,mean_balanced_accuracy,mean_brier,mean_best_iteration,median_best_iteration,log_loss_cutoff,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,22,0.6655,0.6655,59.62%,58.68%,0.2366,64,25,0.670467,3,1.416640,2.625159,0.720908,0.554465,0.049555,0.666938,1.032890,0.008786
1,21,0.6675,0.6675,59.29%,59.20%,0.2376,54,14,0.670467,3,1.031972,2.591255,0.732149,0.603324,0.047888,1.030835,1.010468,0.007651
2,17,0.6605,0.6605,59.05%,58.15%,0.2343,62,12,0.670467,4,2.260454,1.857087,0.727676,0.772640,0.048492,0.512588,1.064875,0.007335
3,18,0.6684,0.6684,59.05%,58.32%,0.2379,42,10,0.670467,4,1.070178,2.475553,0.748674,0.592791,0.049794,1.554391,1.043063,0.006913
4,19,0.6675,0.6675,57.08%,56.49%,0.2375,47,17,0.670467,3,1.022417,2.665636,0.746679,0.558282,0.046173,0.846720,1.035041,0.007180


In [19]:
optuna_model = fit_best_xgb_classifier(
    X_dev=X_dev,
    y_dev=y_dev,
    sample_weight_dates=df_dev['GAME_DATE'],
    sample_weight_lambda=best_trial_min_loss.params.get('sample_weight_lambda'),
    trial=best_trial_min_loss,
)

y_proba_test_optuna = optuna_model.predict_proba(X_test_final)[:, 1]

print('Final holdout metrics')
final_summary_optuna = print_final_metrics(
    'Optuna-selected XGBoost classifier',
    y_test_final,
    y_proba_test_optuna,
)


Final holdout metrics


,model,n_games,accuracy_pct,balanced_accuracy_pct,brier_score,log_loss,mean_predicted_over_pct,mean_confidence_pct
0,Optuna-selected XGBoost classifier,263,50.5703,50.9719,0.2562,0.7064,44.8655,58.3303


In [20]:
threshold_summary_optuna = summarize_confidence_thresholds(
    y_true=y_test_final,
    y_proba=y_proba_test_optuna,
    thresholds=CONFIDENCE_THRESHOLDS,
)

display(
    threshold_summary_optuna.style.format(
        {'pct_of_test': '{:.1%}', 'bet_accuracy_pct': '{:.2f}', 'mean_confidence_pct': '{:.2f}'}
    )
)


,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,0.500000,263,100.0%,50.57,58.33
1,0.520000,215,81.7%,51.16,59.96
2,0.550000,169,64.3%,52.07,61.69
3,0.580000,128,48.7%,51.56,63.28
4,0.600000,94,35.7%,56.38,64.85
5,0.650000,35,13.3%,57.14,68.76


In [21]:
def fit_and_predict_optuna_day_by_day(train_df, test_df):
    train_rows = train_df['__row_id__'].to_numpy(dtype=int)
    test_rows = test_df['__row_id__'].to_numpy(dtype=int)

    X_train = X_all.iloc[train_rows]
    y_train = y_all.iloc[train_rows]
    X_test = X_all.iloc[test_rows]

    model = fit_best_xgb_classifier(
        X_dev=X_train,
        y_dev=y_train,
        sample_weight_dates=train_df['GAME_DATE'],
        sample_weight_lambda=best_trial_min_loss.params.get('sample_weight_lambda'),
        trial=best_trial_min_loss,
    )
    return model.predict_proba(X_test)[:, 1]


day_by_day_optuna, day_by_day_optuna_thresholds = run_day_by_day_classifier_evaluation(
    label='Optuna-selected XGBoost classifier',
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_optuna_day_by_day,
    max_games=TRAIN_GAMES,
)


Optuna-selected XGBoost classifier mean day-by-day Accuracy: 55.94%


,date,train_n_games,test_n_games,train_start_date,train_end_date,Accuracy
0,2026-03-04 00:00:00,2500,5,2024-02-13 00:00:00,2026-03-03 00:00:00,60.00%
1,2026-03-05 00:00:00,2500,9,2024-02-14 00:00:00,2026-03-04 00:00:00,66.67%
2,2026-03-06 00:00:00,2500,7,2024-02-14 00:00:00,2026-03-05 00:00:00,28.57%
3,2026-03-07 00:00:00,2500,5,2024-02-23 00:00:00,2026-03-06 00:00:00,80.00%
4,2026-03-08 00:00:00,2500,8,2024-02-24 00:00:00,2026-03-07 00:00:00,50.00%
5,2026-03-09 00:00:00,2500,5,2024-02-25 00:00:00,2026-03-08 00:00:00,60.00%
6,2026-03-10 00:00:00,2500,10,2024-02-26 00:00:00,2026-03-09 00:00:00,60.00%
7,2026-03-11 00:00:00,2500,6,2024-02-27 00:00:00,2026-03-10 00:00:00,83.33%
8,2026-03-12 00:00:00,2500,8,2024-02-28 00:00:00,2026-03-11 00:00:00,62.50%
9,2026-03-13 00:00:00,2500,8,2024-02-29 00:00:00,2026-03-12 00:00:00,62.50%


Optuna-selected XGBoost classifier thresholded walk-forward accuracy


,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,0.500000,263,100.0%,54.75,58.63
1,0.520000,234,89.0%,54.70,59.55
2,0.550000,181,68.8%,57.46,61.27
3,0.580000,124,47.1%,61.29,63.48
4,0.600000,96,36.5%,60.42,64.81
5,0.650000,42,16.0%,47.62,68.25


In [22]:
optuna_cv_summary = pd.DataFrame(
    [
        {
            'model': 'Optuna-selected XGBoost classifier',
            'validation_accuracy_pct': 100.0 * float(best_trial_min_loss.user_attrs.get('mean_accuracy', np.nan)),
            'validation_balanced_accuracy_pct': 100.0 * float(best_trial_min_loss.user_attrs.get('mean_balanced_accuracy', np.nan)),
            'validation_brier_score': float(best_trial_min_loss.user_attrs.get('mean_brier', np.nan)),
            'validation_log_loss': float(best_trial_min_loss.user_attrs.get('mean_log_loss', np.nan)),
        }
    ]
)

print('Cross-validation comparison with Optuna')
display(
    pd.concat(
        [baseline_cv_summary, cv_classifier_summary, optuna_cv_summary],
        ignore_index=True,
    ).round(4)
)

print('Final holdout comparison with Optuna')
display(
    pd.concat(
        [baseline_final_summary, final_classifier_summary, final_summary_optuna],
        ignore_index=True,
    ).round(4)
)


Cross-validation comparison with Optuna


,model,validation_accuracy_pct,validation_balanced_accuracy_pct,validation_brier_score,validation_log_loss
0,BASE_AVG_ALL_6_ERR,54.7393,52.2378,NaN,NaN
1,BASE_MAJORITY_TOTAL_ONLY_ERR,55.4670,53.7951,NaN,NaN
2,BASE_MAJORITY_LINE_ERROR_ONLY_ERR,53.6255,51.3590,NaN,NaN
3,BASE_MAJORITY_ALL_6_TIE_WITH_PRED_LINE_ERROR_F...,55.9866,53.2463,NaN,NaN
4,XGBoost classifier no sample weights,54.2199,51.0283,0.2485,0.6901
5,XGBoost classifier with sample weights,55.3726,53.0129,0.2486,0.6903
6,Optuna-selected XGBoost classifier,59.0549,58.1472,0.2343,0.6605


Final holdout comparison with Optuna


,model,n_games,accuracy_pct,balanced_accuracy_pct,brier_score,log_loss,mean_predicted_over_pct,mean_confidence_pct
0,BASE_AVG_ALL_6_ERR,263,50.5703,50.8417,NaN,NaN,NaN,NaN
1,BASE_MAJORITY_TOTAL_ONLY_ERR,263,51.3308,51.6025,NaN,NaN,NaN,NaN
2,BASE_MAJORITY_LINE_ERROR_ONLY_ERR,263,53.2319,53.4103,NaN,NaN,NaN,NaN
3,BASE_MAJORITY_ALL_6_TIE_WITH_PRED_LINE_ERROR_F...,263,52.8517,53.0661,NaN,NaN,NaN,NaN
4,XGBoost classifier no sample weights,263,50.1901,50.9459,0.2500,0.6931,48.6108,51.5143
5,XGBoost classifier with sample weights,263,48.6692,49.5401,0.2505,0.6942,47.8655,52.1825
6,Optuna-selected XGBoost classifier,263,50.5703,50.9719,0.2562,0.7064,44.8655,58.3303


In [23]:
df_to_train_split_rows = df_model.copy().tail(TRAIN_GAMES)

X_full = X_all.loc[df_to_train_split_rows.index].copy()
y_full = y_all.loc[df_to_train_split_rows.index].copy()
sample_weight_dates_full = df_to_train_split_rows['GAME_DATE']

production_model = fit_best_xgb_classifier(
    X_dev=X_full,
    y_dev=y_full,
    sample_weight_dates=sample_weight_dates_full,
    sample_weight_lambda=best_trial_min_loss.params.get('sample_weight_lambda'),
    trial=best_trial_min_loss,
)

latest_training_date = pd.to_datetime(df_to_train_split_rows['GAME_DATE']).max()
model_version = latest_training_date.strftime('%d_%m_%y')
model_name = f'meta_learner_xgb_classifier_{model_version}'

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type='meta_learner_classifier',
        prediction_source='meta_learner_xgb_classifier',
        training_code_tag='1.0',
    ),
    training_metrics=TrainingMetrics(
        best_params=best_trial_min_loss.params,
        selected_trial_number=best_trial_min_loss.number,
        mean_best_iteration=best_trial_min_loss.user_attrs.get('mean_best_iteration'),
        median_best_iteration=best_trial_min_loss.user_attrs.get('median_best_iteration'),
        train_games=TRAIN_GAMES,
        sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
        cv_mae=float(best_trial_min_loss.user_attrs.get('mean_log_loss', best_trial_min_loss.value)),
        cv_rmse=best_trial_min_loss.user_attrs.get('mean_brier'),
        cv_ou_acc=best_trial_min_loss.user_attrs.get('mean_accuracy'),
        final_test_mae=float(final_summary_optuna.iloc[0]['log_loss']),
        final_test_rmse=float(final_summary_optuna.iloc[0]['brier_score']),
        final_test_ou_acc=float(final_summary_optuna.iloc[0]['accuracy_pct'] / 100.0),
        nan_threshold=0.0,
        max_na_per_row=0,
        train_date_min=df_to_train_split_rows['GAME_DATE'].min().to_pydatetime(),
        train_date_max=df_to_train_split_rows['GAME_DATE'].max().to_pydatetime(),
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir=MODEL_OUT_DIR,
    metadata=metadata,
)

print(
    f'Production model trained on {len(X_full)} rows using fixed n_estimators from median_best_iteration.'
)
print('Saved model :', model_path)
print('Saved metadata:', meta_path)


Production model trained on 2500 rows using fixed n_estimators from median_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/meta_learner/classifier/meta_learner_xgb_classifier_08_04_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/meta_learner/classifier/meta_learner_xgb_classifier_08_04_26.meta.json


## Logistic Regression Benchmark


In [24]:
from collections.abc import Callable
from typing import Any

from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

CALIBRATION_METHODS = ('raw', 'sigmoid', 'isotonic')
CALIBRATION_N_BINS = 10
MAX_CALIBRATION_INNER_FOLDS = 5
CALIBRATION_FALLBACK_TEST_DATE_FRAC = 0.2
RUN_CALIBRATED_DAY_BY_DAY = True
CALIBRATED_DAY_BY_DAY_VARIANTS = (
    ('Logistic', 'sigmoid'),
    ('XGBoost Optuna', 'sigmoid'),
)


In [25]:
def make_logistic_meta_classifier() -> Pipeline:
    """Build a regularized logistic-regression baseline with safe preprocessing."""
    return Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            (
                'classifier',
                LogisticRegression(
                    C=1.0,
                    penalty='l2',
                    solver='lbfgs',
                    max_iter=5000,
                    random_state=16,
                ),
            ),
        ]
    )


logistic_clf = make_logistic_meta_classifier()

cv_results_logistic = cross_validate(
    logistic_clf,
    X_dev,
    y_dev,
    cv=splits,
    scoring=SCORING,
    return_train_score=True,
    n_jobs=1,
)

print('Logistic regression meta-classifier')
print_classifier_cv_metrics(cv_results_logistic, SCORING)

logistic_clf.fit(X_dev, y_dev)
y_proba_test_logistic = logistic_clf.predict_proba(X_test_final)[:, 1]

print('Final holdout metrics')
final_summary_logistic = print_final_metrics(
    'Logistic raw',
    y_test_final,
    y_proba_test_logistic,
)

threshold_summary_logistic = summarize_confidence_thresholds(
    y_true=y_test_final,
    y_proba=y_proba_test_logistic,
    thresholds=CONFIDENCE_THRESHOLDS,
)

display(
    threshold_summary_logistic.style.format(
        {'pct_of_test': '{:.1%}', 'bet_accuracy_pct': '{:.2f}', 'mean_confidence_pct': '{:.2f}'}
    )
)


Logistic regression meta-classifier
Train Accuracy: 74.57%
Validation Accuracy: 47.38%

Train Balanced_Accuracy: 74.53%
Validation Balanced_Accuracy: 50.21%

Train Brier: 0.16915
Validation Brier: 0.38206

Train LogLoss: 0.50638
Validation LogLoss: 1.46610

Final holdout metrics


,model,n_games,accuracy_pct,balanced_accuracy_pct,brier_score,log_loss,mean_predicted_over_pct,mean_confidence_pct
0,Logistic raw,263,52.8517,52.6322,0.3042,0.8815,56.8086,73.7095


,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,0.500000,263,100.0%,52.85,73.71
1,0.520000,252,95.8%,53.57,74.70
2,0.550000,237,90.1%,53.59,76.05
3,0.580000,221,84.0%,54.75,77.47
4,0.600000,210,79.8%,54.29,78.46
5,0.650000,185,70.3%,54.59,80.63


## Leakage-Free Calibration Helpers


In [26]:
def fit_logistic_meta_classifier(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    train_dates: pd.Series | None = None,
):
    model = make_logistic_meta_classifier()
    model.fit(X_train, y_train)
    return model



def fit_fixed_xgb_meta_classifier(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    train_dates: pd.Series | None = None,
):
    model = XGBClassifier(**XGB_FIXED_PARAMS)
    model.fit(X_train, y_train)
    return model



def fit_optuna_selected_xgb_meta_classifier(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    train_dates: pd.Series | None = None,
):
    if train_dates is None:
        raise ValueError('train_dates are required for the Optuna-selected XGBoost model.')
    return fit_best_xgb_classifier(
        X_dev=X_train,
        y_dev=y_train,
        sample_weight_dates=train_dates,
        sample_weight_lambda=best_trial_min_loss.params.get('sample_weight_lambda'),
        trial=best_trial_min_loss,
    )


MODEL_FITTERS: dict[str, Callable[[pd.DataFrame, pd.Series, pd.Series | None], Any]] = {
    'Logistic': fit_logistic_meta_classifier,
    'XGBoost fixed': fit_fixed_xgb_meta_classifier,
    'XGBoost Optuna': fit_optuna_selected_xgb_meta_classifier,
}



def predict_positive_probability(model, X: pd.DataFrame) -> np.ndarray:
    probabilities = model.predict_proba(X)
    if probabilities.ndim != 2 or probabilities.shape[1] < 2:
        raise ValueError('Model must return two-class probabilities.')
    return np.asarray(probabilities[:, 1], dtype=float)



def safe_logit(probabilities: pd.Series | np.ndarray, eps: float = 1e-6) -> np.ndarray:
    clipped = np.clip(np.asarray(probabilities, dtype=float), eps, 1.0 - eps)
    return np.log(clipped / (1.0 - clipped))



def make_model_variant_label(model_family: str, calibration_method: str) -> str:
    if calibration_method == 'raw':
        return f'{model_family} raw'
    return f'{model_family} + {calibration_method}'



def build_fallback_inner_split(
    df_train: pd.DataFrame,
    *,
    test_date_frac: float = CALIBRATION_FALLBACK_TEST_DATE_FRAC,
) -> tuple[list[tuple[np.ndarray, np.ndarray]], pd.DataFrame]:
    """Build one leakage-free latest-dates split when anchored inner folds are not feasible."""
    dates = pd.to_datetime(df_train['GAME_DATE'], errors='coerce').dt.normalize()
    unique_dates = pd.Index(sorted(dates.dropna().unique()))
    if len(unique_dates) < 2:
        raise ValueError('Not enough unique dates to build a fallback inner calibration split.')

    n_test_dates = max(1, int(np.ceil(len(unique_dates) * test_date_frac)))
    test_dates = set(unique_dates[-n_test_dates:])
    test_mask = dates.isin(test_dates).to_numpy()
    train_idx = np.flatnonzero(~test_mask)
    test_idx = np.flatnonzero(test_mask)

    if len(train_idx) == 0 or len(test_idx) == 0:
        raise ValueError('Fallback calibration split produced an empty train or test partition.')

    fold_info = pd.DataFrame(
        [
            {
                'fold': 1,
                'train_n_games': int(len(train_idx)),
                'test_n_games': int(len(test_idx)),
                'train_start_date': dates.iloc[train_idx].min(),
                'train_end_date': dates.iloc[train_idx].max(),
                'test_start_date': dates.iloc[test_idx].min(),
                'test_end_date': dates.iloc[test_idx].max(),
            }
        ]
    )
    return [(train_idx, test_idx)], fold_info



def build_inner_calibration_splits(
    df_train: pd.DataFrame,
    *,
    max_folds: int = MAX_CALIBRATION_INNER_FOLDS,
) -> tuple[list[tuple[np.ndarray, np.ndarray]], pd.DataFrame]:
    """Create conservative time-aware inner splits for calibrator fitting."""
    n_rows = len(df_train)
    inner_test_games = max(10, min(TEST_GAMES, max(10, n_rows // 8)))
    inner_min_train_games = min(MIN_TRAIN_GAMES, max(60, inner_test_games * 3))

    try:
        return make_test_anchored_walk_forward_splits(
            df=df_train,
            date_col='GAME_DATE',
            season_col='SEASON_YEAR',
            test_games=inner_test_games,
            step_games_between_tests=inner_test_games,
            train_games=None,
            min_train_games=inner_min_train_games,
            max_folds=max(1, max_folds),
            verbose=0,
        )
    except ValueError:
        return build_fallback_inner_split(df_train)



def generate_time_aware_oof_probabilities(
    *,
    df_train: pd.DataFrame,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    fit_model_fn: Callable[[pd.DataFrame, pd.Series, pd.Series | None], Any],
    inner_splits: list[tuple[np.ndarray, np.ndarray]] | None = None,
) -> pd.DataFrame:
    """Generate leakage-free OOF probabilities on a training subset for calibration."""
    local_df = df_train.reset_index(drop=True).copy()
    local_X = X_train.reset_index(drop=True).copy()
    local_y = pd.Series(y_train).reset_index(drop=True).astype(int)

    if inner_splits is None:
        inner_splits, _ = build_inner_calibration_splits(local_df)

    rows: list[pd.DataFrame] = []
    for inner_fold, (inner_train_idx, inner_valid_idx) in enumerate(inner_splits, start=1):
        model = fit_model_fn(
            local_X.iloc[inner_train_idx].copy(),
            local_y.iloc[inner_train_idx].copy(),
            local_df.iloc[inner_train_idx]['GAME_DATE'].copy(),
        )
        valid_proba = predict_positive_probability(
            model,
            local_X.iloc[inner_valid_idx].copy(),
        )
        rows.append(
            pd.DataFrame(
                {
                    'inner_fold': inner_fold,
                    'row_id': inner_valid_idx,
                    'GAME_DATE': local_df.iloc[inner_valid_idx]['GAME_DATE'].to_numpy(),
                    'y_true': local_y.iloc[inner_valid_idx].to_numpy(dtype=int),
                    'y_proba_raw': valid_proba,
                }
            )
        )

    if not rows:
        raise ValueError('No inner OOF predictions were generated for calibration.')

    return pd.concat(rows, ignore_index=True).sort_values('row_id').reset_index(drop=True)



def fit_probability_calibrator(
    *,
    y_true: pd.Series | np.ndarray,
    y_proba: pd.Series | np.ndarray,
    method: str,
) -> dict[str, Any]:
    """Fit a one-dimensional probability calibrator on leakage-free OOF predictions."""
    y_true_arr = np.asarray(y_true, dtype=int)
    y_proba_arr = np.clip(np.asarray(y_proba, dtype=float), 1e-6, 1.0 - 1e-6)

    if method == 'raw':
        return {'requested_method': method, 'effective_method': 'raw', 'model': None}

    if np.unique(y_true_arr).size < 2:
        return {
            'requested_method': method,
            'effective_method': 'raw',
            'model': None,
            'fallback_reason': 'single_class_calibration_sample',
        }

    if method == 'sigmoid':
        calibrator = LogisticRegression(
            C=1e6,
            solver='lbfgs',
            max_iter=5000,
            random_state=16,
        )
        calibrator.fit(safe_logit(y_proba_arr).reshape(-1, 1), y_true_arr)
        return {'requested_method': method, 'effective_method': 'sigmoid', 'model': calibrator}

    if method == 'isotonic':
        if np.unique(y_proba_arr).size < 2:
            return {
                'requested_method': method,
                'effective_method': 'raw',
                'model': None,
                'fallback_reason': 'constant_calibration_scores',
            }
        calibrator = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
        calibrator.fit(y_proba_arr, y_true_arr)
        return {'requested_method': method, 'effective_method': 'isotonic', 'model': calibrator}

    raise ValueError(f'Unsupported calibration method: {method}')



def apply_probability_calibrator(
    calibrator: dict[str, Any],
    y_proba: pd.Series | np.ndarray,
) -> np.ndarray:
    """Apply a fitted calibrator to raw probabilities."""
    raw = np.clip(np.asarray(y_proba, dtype=float), 1e-6, 1.0 - 1e-6)
    effective_method = calibrator.get('effective_method', 'raw')

    if effective_method == 'raw':
        return raw
    if effective_method == 'sigmoid':
        fitted_model = calibrator['model']
        return np.clip(
            fitted_model.predict_proba(safe_logit(raw).reshape(-1, 1))[:, 1],
            0.0,
            1.0,
        )
    if effective_method == 'isotonic':
        fitted_model = calibrator['model']
        return np.clip(fitted_model.predict(raw), 0.0, 1.0)

    raise ValueError(f'Unsupported effective calibration method: {effective_method}')



def expected_calibration_error(
    y_true: pd.Series | np.ndarray,
    y_proba: pd.Series | np.ndarray,
    *,
    n_bins: int = CALIBRATION_N_BINS,
) -> float:
    y_true_arr = np.asarray(y_true, dtype=int)
    y_proba_arr = np.asarray(y_proba, dtype=float)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.clip(np.digitize(y_proba_arr, bin_edges, right=True) - 1, 0, n_bins - 1)

    ece = 0.0
    n_total = len(y_true_arr)
    for bin_id in range(n_bins):
        mask = bin_ids == bin_id
        if not np.any(mask):
            continue
        prob_mean = float(np.mean(y_proba_arr[mask]))
        outcome_mean = float(np.mean(y_true_arr[mask]))
        ece += (mask.sum() / n_total) * abs(prob_mean - outcome_mean)
    return float(ece)



def build_calibration_table(
    *,
    y_true: pd.Series | np.ndarray,
    y_proba: pd.Series | np.ndarray,
    n_bins: int = CALIBRATION_N_BINS,
) -> pd.DataFrame:
    y_true_arr = np.asarray(y_true, dtype=int)
    y_proba_arr = np.asarray(y_proba, dtype=float)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.clip(np.digitize(y_proba_arr, bin_edges, right=True) - 1, 0, n_bins - 1)

    rows: list[dict[str, float | int]] = []
    n_total = len(y_true_arr)
    for bin_id in range(n_bins):
        left = float(bin_edges[bin_id])
        right = float(bin_edges[bin_id + 1])
        mask = bin_ids == bin_id
        n_bin = int(mask.sum())
        rows.append(
            {
                'bin_id': bin_id,
                'bin_left': left,
                'bin_right': right,
                'bin_label': f'[{left:.2f}, {right:.2f}]',
                'n_games': n_bin,
                'pct_of_sample': (n_bin / n_total) if n_total else np.nan,
                'mean_predicted_over_pct': (100.0 * float(np.mean(y_proba_arr[mask]))) if n_bin else np.nan,
                'empirical_over_rate_pct': (100.0 * float(np.mean(y_true_arr[mask]))) if n_bin else np.nan,
                'calibration_gap_pct': (
                    100.0 * float(np.mean(y_proba_arr[mask]) - np.mean(y_true_arr[mask]))
                    if n_bin
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(rows)



def summarize_probability_row(
    *,
    y_true: pd.Series | np.ndarray,
    y_proba: pd.Series | np.ndarray,
    label: str,
) -> dict[str, float | int | str]:
    row = summarize_classifier_predictions(
        y_true=y_true,
        y_proba=y_proba,
        label=label,
    )
    row['ece_10_pct'] = 100.0 * expected_calibration_error(y_true, y_proba)
    return row



def summarize_cv_probability_predictions(predictions_df: pd.DataFrame) -> pd.DataFrame:
    fold_rows: list[dict[str, float | int | str]] = []
    for (model, fold), group in predictions_df.groupby(['model', 'fold'], sort=True):
        row = summarize_probability_row(
            y_true=group['y_true'],
            y_proba=group['y_proba'],
            label=model,
        )
        row['fold'] = int(fold)
        fold_rows.append(row)

    fold_df = pd.DataFrame(fold_rows)
    if fold_df.empty:
        return pd.DataFrame()

    summary = (
        fold_df.groupby('model', as_index=False)
        .agg(
            validation_accuracy_pct=('accuracy_pct', 'mean'),
            validation_balanced_accuracy_pct=('balanced_accuracy_pct', 'mean'),
            validation_brier_score=('brier_score', 'mean'),
            validation_log_loss=('log_loss', 'mean'),
            validation_ece_10_pct=('ece_10_pct', 'mean'),
            validation_n_games=('n_games', 'sum'),
            n_folds=('fold', 'nunique'),
        )
        .sort_values('validation_log_loss', ascending=True)
        .reset_index(drop=True)
    )
    return summary



def summarize_probability_predictions_by_model(predictions_df: pd.DataFrame) -> pd.DataFrame:
    rows = [
        summarize_probability_row(
            y_true=group['y_true'],
            y_proba=group['y_proba'],
            label=model,
        )
        for model, group in predictions_df.groupby('model', sort=False)
    ]
    return pd.DataFrame(rows)



def summarize_thresholds_by_model(
    predictions_df: pd.DataFrame,
    *,
    thresholds: tuple[float, ...] = CONFIDENCE_THRESHOLDS,
) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for model, group in predictions_df.groupby('model', sort=False):
        threshold_df = summarize_confidence_thresholds(
            y_true=group['y_true'],
            y_proba=group['y_proba'],
            thresholds=thresholds,
        ).copy()
        threshold_df.insert(0, 'model', model)
        frames.append(threshold_df)
    return pd.concat(frames, ignore_index=True)



def summarize_confidence_band_calibration(
    *,
    y_true: pd.Series | np.ndarray,
    y_proba: pd.Series | np.ndarray,
    thresholds: tuple[float, ...] = CONFIDENCE_THRESHOLDS,
) -> pd.DataFrame:
    y_true_arr = np.asarray(y_true, dtype=int)
    y_proba_arr = np.asarray(y_proba, dtype=float)
    hard_pred = (y_proba_arr >= 0.5).astype(int)
    selected_class_proba = np.where(hard_pred == 1, y_proba_arr, 1.0 - y_proba_arr)
    correctness = (hard_pred == y_true_arr).astype(int)

    rows: list[dict[str, float | int]] = []
    n_total = len(y_true_arr)
    for threshold in thresholds:
        mask = selected_class_proba >= threshold
        n_games = int(mask.sum())
        mean_prob = np.nan if n_games == 0 else 100.0 * float(np.mean(selected_class_proba[mask]))
        empirical_acc = np.nan if n_games == 0 else 100.0 * float(np.mean(correctness[mask]))
        rows.append(
            {
                'threshold_confidence_gte': threshold,
                'n_games': n_games,
                'pct_of_sample': (n_games / n_total) if n_total else np.nan,
                'mean_selected_class_probability_pct': mean_prob,
                'empirical_accuracy_pct': empirical_acc,
                'calibration_gap_pct': (
                    np.nan if n_games == 0 else float(mean_prob - empirical_acc)
                ),
            }
        )
    return pd.DataFrame(rows)



def summarize_confidence_bands_by_model(
    predictions_df: pd.DataFrame,
    *,
    thresholds: tuple[float, ...] = CONFIDENCE_THRESHOLDS,
) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for model, group in predictions_df.groupby('model', sort=False):
        summary = summarize_confidence_band_calibration(
            y_true=group['y_true'],
            y_proba=group['y_proba'],
            thresholds=thresholds,
        ).copy()
        summary.insert(0, 'model', model)
        frames.append(summary)
    return pd.concat(frames, ignore_index=True)



def build_calibration_tables_by_model(
    predictions_df: pd.DataFrame,
    *,
    n_bins: int = CALIBRATION_N_BINS,
) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for model, group in predictions_df.groupby('model', sort=False):
        table = build_calibration_table(
            y_true=group['y_true'],
            y_proba=group['y_proba'],
            n_bins=n_bins,
        ).copy()
        table.insert(0, 'model', model)
        frames.append(table)
    return pd.concat(frames, ignore_index=True)



def collect_outer_fold_calibrated_predictions(
    *,
    model_family: str,
    fit_model_fn: Callable[[pd.DataFrame, pd.Series, pd.Series | None], Any],
    df_dev: pd.DataFrame,
    X_dev: pd.DataFrame,
    y_dev: pd.Series,
    splits: list[tuple[np.ndarray, np.ndarray]],
    calibration_methods: tuple[str, ...] = CALIBRATION_METHODS,
) -> pd.DataFrame:
    """Evaluate raw and calibrated probabilities on the outer development splits."""
    rows: list[pd.DataFrame] = []
    local_df_dev = df_dev.reset_index(drop=True).copy()
    local_X_dev = X_dev.reset_index(drop=True).copy()
    local_y_dev = pd.Series(y_dev).reset_index(drop=True).astype(int)

    for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
        outer_train_df = local_df_dev.iloc[train_idx].reset_index(drop=True).copy()
        outer_valid_df = local_df_dev.iloc[valid_idx].reset_index(drop=True).copy()
        X_outer_train = local_X_dev.iloc[train_idx].reset_index(drop=True).copy()
        y_outer_train = local_y_dev.iloc[train_idx].reset_index(drop=True).copy()
        X_outer_valid = local_X_dev.iloc[valid_idx].reset_index(drop=True).copy()
        y_outer_valid = local_y_dev.iloc[valid_idx].reset_index(drop=True).copy()

        calibration_oof = generate_time_aware_oof_probabilities(
            df_train=outer_train_df,
            X_train=X_outer_train,
            y_train=y_outer_train,
            fit_model_fn=fit_model_fn,
        )
        calibrators = {
            method: fit_probability_calibrator(
                y_true=calibration_oof['y_true'],
                y_proba=calibration_oof['y_proba_raw'],
                method=method,
            )
            for method in calibration_methods
            if method != 'raw'
        }

        fitted_model = fit_model_fn(
            X_outer_train,
            y_outer_train,
            outer_train_df['GAME_DATE'],
        )
        raw_valid_proba = predict_positive_probability(fitted_model, X_outer_valid)

        for method in calibration_methods:
            calibrator = (
                {'requested_method': 'raw', 'effective_method': 'raw', 'model': None}
                if method == 'raw'
                else calibrators[method]
            )
            calibrated_valid_proba = apply_probability_calibrator(calibrator, raw_valid_proba)
            fold_frame = pd.DataFrame(
                {
                    'dataset': 'development_cv',
                    'model_family': model_family,
                    'calibration_method': method,
                    'effective_calibration_method': calibrator.get('effective_method', method),
                    'model': make_model_variant_label(model_family, method),
                    'fold': fold,
                    'GAME_DATE': outer_valid_df['GAME_DATE'].to_numpy(),
                    'y_true': y_outer_valid.to_numpy(dtype=int),
                    'y_proba_raw': raw_valid_proba,
                    'y_proba': calibrated_valid_proba,
                    'calibration_oof_rows': int(len(calibration_oof)),
                }
            )
            rows.append(fold_frame)

    return pd.concat(rows, ignore_index=True)



def collect_final_holdout_calibrated_predictions(
    *,
    model_family: str,
    fit_model_fn: Callable[[pd.DataFrame, pd.Series, pd.Series | None], Any],
    df_dev: pd.DataFrame,
    X_dev: pd.DataFrame,
    y_dev: pd.Series,
    df_test_final: pd.DataFrame,
    X_test_final: pd.DataFrame,
    y_test_final: pd.Series,
    calibration_methods: tuple[str, ...] = CALIBRATION_METHODS,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fit calibrators on development OOF predictions, then score final holdout."""
    local_df_dev = df_dev.reset_index(drop=True).copy()
    local_X_dev = X_dev.reset_index(drop=True).copy()
    local_y_dev = pd.Series(y_dev).reset_index(drop=True).astype(int)

    calibration_oof = generate_time_aware_oof_probabilities(
        df_train=local_df_dev,
        X_train=local_X_dev,
        y_train=local_y_dev,
        fit_model_fn=fit_model_fn,
        inner_splits=splits,
    )
    calibrators = {
        method: fit_probability_calibrator(
            y_true=calibration_oof['y_true'],
            y_proba=calibration_oof['y_proba_raw'],
            method=method,
        )
        for method in calibration_methods
        if method != 'raw'
    }

    fitted_model = fit_model_fn(local_X_dev, local_y_dev, local_df_dev['GAME_DATE'])
    raw_test_proba = predict_positive_probability(fitted_model, X_test_final)

    frames: list[pd.DataFrame] = []
    for method in calibration_methods:
        calibrator = (
            {'requested_method': 'raw', 'effective_method': 'raw', 'model': None}
            if method == 'raw'
            else calibrators[method]
        )
        calibrated_test_proba = apply_probability_calibrator(calibrator, raw_test_proba)
        frames.append(
            pd.DataFrame(
                {
                    'dataset': 'final_holdout',
                    'model_family': model_family,
                    'calibration_method': method,
                    'effective_calibration_method': calibrator.get('effective_method', method),
                    'model': make_model_variant_label(model_family, method),
                    'GAME_DATE': df_test_final['GAME_DATE'].to_numpy(),
                    'y_true': pd.Series(y_test_final).to_numpy(dtype=int),
                    'y_proba_raw': raw_test_proba,
                    'y_proba': calibrated_test_proba,
                    'calibration_oof_rows': int(len(calibration_oof)),
                }
            )
        )

    return pd.concat(frames, ignore_index=True), calibration_oof



def make_calibrated_day_by_day_predict_fn(
    *,
    fit_model_fn: Callable[[pd.DataFrame, pd.Series, pd.Series | None], Any],
    calibration_method: str,
):
    """Create a leakage-free day-by-day prediction function with calibration."""
    def fit_and_predict(train_df: pd.DataFrame, test_df: pd.DataFrame) -> np.ndarray:
        train_rows = train_df['__row_id__'].to_numpy(dtype=int)
        test_rows = test_df['__row_id__'].to_numpy(dtype=int)

        local_train_df = train_df.reset_index(drop=True).copy()
        X_train = X_all.iloc[train_rows].reset_index(drop=True).copy()
        y_train = y_all.iloc[train_rows].reset_index(drop=True).copy()
        X_test = X_all.iloc[test_rows].copy()

        calibration_oof = generate_time_aware_oof_probabilities(
            df_train=local_train_df,
            X_train=X_train,
            y_train=y_train,
            fit_model_fn=fit_model_fn,
        )
        calibrator = fit_probability_calibrator(
            y_true=calibration_oof['y_true'],
            y_proba=calibration_oof['y_proba_raw'],
            method=calibration_method,
        )
        fitted_model = fit_model_fn(X_train, y_train, local_train_df['GAME_DATE'])
        raw_test_proba = predict_positive_probability(fitted_model, X_test)
        return apply_probability_calibrator(calibrator, raw_test_proba)

    return fit_and_predict


## Calibration Experiments On Development CV


In [27]:
calibration_cv_predictions = pd.concat(
    [
        collect_outer_fold_calibrated_predictions(
            model_family=model_family,
            fit_model_fn=fit_model_fn,
            df_dev=df_dev,
            X_dev=X_dev,
            y_dev=y_dev,
            splits=splits,
        )
        for model_family, fit_model_fn in MODEL_FITTERS.items()
    ],
    ignore_index=True,
)

calibration_cv_summary = summarize_cv_probability_predictions(calibration_cv_predictions)
calibration_cv_threshold_summary = summarize_thresholds_by_model(calibration_cv_predictions)
calibration_cv_confidence_band_summary = summarize_confidence_bands_by_model(calibration_cv_predictions)
calibration_cv_tables = build_calibration_tables_by_model(calibration_cv_predictions)

print('Leakage-free development CV summary')
display(calibration_cv_summary.round(4))

print('Leakage-free development CV threshold summary')
display(
    calibration_cv_threshold_summary.style.format(
        {
            'pct_of_test': '{:.1%}',
            'bet_accuracy_pct': '{:.2f}',
            'mean_confidence_pct': '{:.2f}',
        }
    )
)

print('Leakage-free development CV calibration diagnostics around betting thresholds')
display(
    calibration_cv_confidence_band_summary.style.format(
        {
            'pct_of_sample': '{:.1%}',
            'mean_selected_class_probability_pct': '{:.2f}',
            'empirical_accuracy_pct': '{:.2f}',
            'calibration_gap_pct': '{:.2f}',
        }
    )
)

print('Leakage-free development CV reliability table')
display(
    calibration_cv_tables.style.format(
        {
            'pct_of_sample': '{:.1%}',
            'mean_predicted_over_pct': '{:.2f}',
            'empirical_over_rate_pct': '{:.2f}',
            'calibration_gap_pct': '{:.2f}',
        }
    )
)

Leakage-free development CV summary


,model,validation_accuracy_pct,validation_balanced_accuracy_pct,validation_brier_score,validation_log_loss,validation_ece_10_pct,validation_n_games,n_folds
0,XGBoost Optuna raw,55.2192,54.4926,0.2473,0.6877,14.0012,461,14
1,XGBoost fixed raw,54.2199,51.0283,0.2485,0.6901,9.8771,461,14
2,Logistic + sigmoid,49.2195,48.0732,0.2514,0.6964,12.8766,461,14
3,XGBoost fixed + sigmoid,51.5484,50.7094,0.2521,0.6975,10.8160,461,14
4,XGBoost Optuna + sigmoid,50.3683,48.7097,0.2540,0.7015,13.4599,461,14
5,Logistic + isotonic,51.0473,51.3555,0.2563,0.7877,12.3722,461,14
6,XGBoost Optuna + isotonic,52.3139,51.9066,0.2507,0.8396,12.4157,461,14
7,XGBoost fixed + isotonic,51.0523,51.2823,0.2582,1.1785,13.4465,461,14
8,Logistic raw,47.3765,50.2109,0.3821,1.4041,37.9658,461,14


Leakage-free development CV threshold summary


,model,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,Logistic raw,0.500000,461,100.0%,47.72,80.64
1,Logistic raw,0.520000,449,97.4%,47.66,81.43
2,Logistic raw,0.550000,430,93.3%,47.91,82.67
3,Logistic raw,0.580000,414,89.8%,48.07,83.67
4,Logistic raw,0.600000,390,84.6%,48.21,85.19
5,Logistic raw,0.650000,361,78.3%,47.65,87.01
6,Logistic + sigmoid,0.500000,461,100.0%,48.81,56.09
7,Logistic + sigmoid,0.520000,373,80.9%,50.94,57.32
8,Logistic + sigmoid,0.550000,251,54.4%,52.99,59.23
9,Logistic + sigmoid,0.580000,92,20.0%,56.52,64.16


Leakage-free development CV calibration diagnostics around betting thresholds


,model,threshold_confidence_gte,n_games,pct_of_sample,mean_selected_class_probability_pct,empirical_accuracy_pct,calibration_gap_pct
0,Logistic raw,0.500000,461,100.0%,80.64,47.72,32.92
1,Logistic raw,0.520000,449,97.4%,81.43,47.66,33.77
2,Logistic raw,0.550000,430,93.3%,82.67,47.91,34.76
3,Logistic raw,0.580000,414,89.8%,83.67,48.07,35.61
4,Logistic raw,0.600000,390,84.6%,85.19,48.21,36.98
5,Logistic raw,0.650000,361,78.3%,87.01,47.65,39.36
6,Logistic + sigmoid,0.500000,461,100.0%,56.09,48.81,7.28
7,Logistic + sigmoid,0.520000,373,80.9%,57.32,50.94,6.38
8,Logistic + sigmoid,0.550000,251,54.4%,59.23,52.99,6.24
9,Logistic + sigmoid,0.580000,92,20.0%,64.16,56.52,7.64


Leakage-free development CV reliability table


,model,bin_id,bin_left,bin_right,bin_label,n_games,pct_of_sample,mean_predicted_over_pct,empirical_over_rate_pct,calibration_gap_pct
0,Logistic raw,0,0.000000,0.100000,"[0.00, 0.10]",96,20.8%,4.63,46.88,-42.25
1,Logistic raw,1,0.100000,0.200000,"[0.10, 0.20]",39,8.5%,14.61,38.46,-23.85
2,Logistic raw,2,0.200000,0.300000,"[0.20, 0.30]",33,7.2%,24.81,63.64,-38.82
3,Logistic raw,3,0.300000,0.400000,"[0.30, 0.40]",35,7.6%,35.43,45.71,-10.28
4,Logistic raw,4,0.400000,0.500000,"[0.40, 0.50]",40,8.7%,44.31,50.00,-5.69
5,Logistic raw,5,0.500000,0.600000,"[0.50, 0.60]",31,6.7%,55.61,38.71,16.90
6,Logistic raw,6,0.600000,0.700000,"[0.60, 0.70]",28,6.1%,65.53,39.29,26.25
7,Logistic raw,7,0.700000,0.800000,"[0.70, 0.80]",34,7.4%,75.35,50.00,25.35
8,Logistic raw,8,0.800000,0.900000,"[0.80, 0.90]",45,9.8%,85.10,40.00,45.10
9,Logistic raw,9,0.900000,1.000000,"[0.90, 1.00]",80,17.4%,97.13,45.00,52.13


## Final Holdout Calibration Experiments


In [28]:
final_holdout_prediction_frames = []
final_holdout_calibration_oof = {}

for model_family, fit_model_fn in MODEL_FITTERS.items():
    holdout_predictions, calibration_oof = collect_final_holdout_calibrated_predictions(
        model_family=model_family,
        fit_model_fn=fit_model_fn,
        df_dev=df_dev,
        X_dev=X_dev,
        y_dev=y_dev,
        df_test_final=df_test_final,
        X_test_final=X_test_final,
        y_test_final=y_test_final,
    )
    final_holdout_prediction_frames.append(holdout_predictions)
    final_holdout_calibration_oof[model_family] = calibration_oof

calibration_holdout_predictions = pd.concat(final_holdout_prediction_frames, ignore_index=True)
calibration_holdout_summary = summarize_probability_predictions_by_model(calibration_holdout_predictions)
calibration_holdout_threshold_summary = summarize_thresholds_by_model(calibration_holdout_predictions)
calibration_holdout_confidence_band_summary = summarize_confidence_bands_by_model(calibration_holdout_predictions)
calibration_holdout_tables = build_calibration_tables_by_model(calibration_holdout_predictions)

print('Final holdout summary for raw and calibrated probabilities')
display(calibration_holdout_summary.round(4))

print('Final holdout threshold summary')
display(
    calibration_holdout_threshold_summary.style.format(
        {
            'pct_of_test': '{:.1%}',
            'bet_accuracy_pct': '{:.2f}',
            'mean_confidence_pct': '{:.2f}',
        }
    )
)

print('Final holdout calibration diagnostics around betting thresholds')
display(
    calibration_holdout_confidence_band_summary.style.format(
        {
            'pct_of_sample': '{:.1%}',
            'mean_selected_class_probability_pct': '{:.2f}',
            'empirical_accuracy_pct': '{:.2f}',
            'calibration_gap_pct': '{:.2f}',
        }
    )
)

print('Final holdout reliability table')
display(
    calibration_holdout_tables.style.format(
        {
            'pct_of_sample': '{:.1%}',
            'mean_predicted_over_pct': '{:.2f}',
            'empirical_over_rate_pct': '{:.2f}',
            'calibration_gap_pct': '{:.2f}',
        }
    )
)

Final holdout summary for raw and calibrated probabilities


,model,n_games,accuracy_pct,balanced_accuracy_pct,brier_score,log_loss,mean_predicted_over_pct,mean_confidence_pct,ece_10_pct
0,Logistic raw,263,52.8517,52.6322,0.3042,0.8815,56.8086,73.7095,20.8578
1,Logistic + sigmoid,263,49.0494,50.0000,0.2527,0.6987,45.7566,54.2434,5.1940
2,Logistic + isotonic,263,49.0494,50.0000,0.2528,0.6988,45.6001,54.3999,5.3504
3,XGBoost fixed raw,263,50.1901,50.9459,0.2500,0.6931,48.6108,51.5143,2.3397
4,XGBoost fixed + sigmoid,263,50.9506,51.7644,0.2518,0.6968,45.7350,54.4511,5.2156
5,XGBoost fixed + isotonic,263,50.5703,51.1888,0.2516,0.6964,46.2194,54.6769,4.7312
6,XGBoost Optuna raw,263,50.5703,50.9719,0.2562,0.7064,44.8655,58.3303,7.7599
7,XGBoost Optuna + sigmoid,263,48.6692,49.3810,0.2557,0.7049,43.4254,57.4472,8.7780
8,XGBoost Optuna + isotonic,263,50.1901,50.8302,0.2671,0.7449,41.0693,60.4385,11.5960


Final holdout threshold summary


,model,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,Logistic raw,0.500000,263,100.0%,52.85,73.71
1,Logistic raw,0.520000,252,95.8%,53.57,74.70
2,Logistic raw,0.550000,237,90.1%,53.59,76.05
3,Logistic raw,0.580000,221,84.0%,54.75,77.47
4,Logistic raw,0.600000,210,79.8%,54.29,78.46
5,Logistic raw,0.650000,185,70.3%,54.59,80.63
6,Logistic + sigmoid,0.500000,263,100.0%,49.05,54.24
7,Logistic + sigmoid,0.520000,263,100.0%,49.05,54.24
8,Logistic + sigmoid,0.550000,0,0.0%,nan,nan
9,Logistic + sigmoid,0.580000,0,0.0%,nan,nan


Final holdout calibration diagnostics around betting thresholds


,model,threshold_confidence_gte,n_games,pct_of_sample,mean_selected_class_probability_pct,empirical_accuracy_pct,calibration_gap_pct
0,Logistic raw,0.500000,263,100.0%,73.71,52.85,20.86
1,Logistic raw,0.520000,252,95.8%,74.70,53.57,21.13
2,Logistic raw,0.550000,237,90.1%,76.05,53.59,22.46
3,Logistic raw,0.580000,221,84.0%,77.47,54.75,22.72
4,Logistic raw,0.600000,210,79.8%,78.46,54.29,24.17
5,Logistic raw,0.650000,185,70.3%,80.63,54.59,26.04
6,Logistic + sigmoid,0.500000,263,100.0%,54.24,49.05,5.19
7,Logistic + sigmoid,0.520000,263,100.0%,54.24,49.05,5.19
8,Logistic + sigmoid,0.550000,0,0.0%,nan,nan,nan
9,Logistic + sigmoid,0.580000,0,0.0%,nan,nan,nan


Final holdout reliability table


,model,bin_id,bin_left,bin_right,bin_label,n_games,pct_of_sample,mean_predicted_over_pct,empirical_over_rate_pct,calibration_gap_pct
0,Logistic raw,0,0.000000,0.100000,"[0.00, 0.10]",15,5.7%,5.75,53.33,-47.58
1,Logistic raw,1,0.100000,0.200000,"[0.10, 0.20]",17,6.5%,15.49,52.94,-37.46
2,Logistic raw,2,0.200000,0.300000,"[0.20, 0.30]",18,6.8%,24.74,38.89,-14.15
3,Logistic raw,3,0.300000,0.400000,"[0.30, 0.40]",27,10.3%,34.69,48.15,-13.46
4,Logistic raw,4,0.400000,0.500000,"[0.40, 0.50]",24,9.1%,45.67,45.83,-0.16
5,Logistic raw,5,0.500000,0.600000,"[0.50, 0.60]",29,11.0%,55.36,41.38,13.98
6,Logistic raw,6,0.600000,0.700000,"[0.60, 0.70]",32,12.2%,65.51,43.75,21.76
7,Logistic raw,7,0.700000,0.800000,"[0.70, 0.80]",40,15.2%,74.56,65.00,9.56
8,Logistic raw,8,0.800000,0.900000,"[0.80, 0.90]",34,12.9%,85.68,52.94,32.74
9,Logistic raw,9,0.900000,1.000000,"[0.90, 1.00]",27,10.3%,93.17,59.26,33.91


## Day-By-Day Calibrated Walk-Forward


In [29]:
day_by_day_calibrated_results: dict[str, Any] = {}

if RUN_CALIBRATED_DAY_BY_DAY:
    for model_family, calibration_method in CALIBRATED_DAY_BY_DAY_VARIANTS:
        fit_and_predict = make_calibrated_day_by_day_predict_fn(
            fit_model_fn=MODEL_FITTERS[model_family],
            calibration_method=calibration_method,
        )
        label = make_model_variant_label(model_family, calibration_method)
        result, threshold_summary = run_day_by_day_classifier_evaluation(
            label=label,
            df_dev=df_dev,
            df_test_final=df_test_final,
            fit_and_predict=fit_and_predict,
            max_games=TRAIN_GAMES,
        )
        day_by_day_calibrated_results[label] = {
            'result': result,
            'threshold_summary': threshold_summary,
            'summary': pd.DataFrame(
                [
                    summarize_probability_row(
                        y_true=result.predictions['y_true'],
                        y_proba=result.predictions['y_pred'],
                        label=label,
                    )
                ]
            ),
        }
else:
    print('Set RUN_CALIBRATED_DAY_BY_DAY = True to run calibrated walk-forward evaluation.')

Logistic + sigmoid mean day-by-day Accuracy: 48.32%


,date,train_n_games,test_n_games,train_start_date,train_end_date,Accuracy
0,2026-03-04 00:00:00,2500,5,2024-02-13 00:00:00,2026-03-03 00:00:00,60.00%
1,2026-03-05 00:00:00,2500,9,2024-02-14 00:00:00,2026-03-04 00:00:00,77.78%
2,2026-03-06 00:00:00,2500,7,2024-02-14 00:00:00,2026-03-05 00:00:00,28.57%
3,2026-03-07 00:00:00,2500,5,2024-02-23 00:00:00,2026-03-06 00:00:00,80.00%
4,2026-03-08 00:00:00,2500,8,2024-02-24 00:00:00,2026-03-07 00:00:00,50.00%
5,2026-03-09 00:00:00,2500,5,2024-02-25 00:00:00,2026-03-08 00:00:00,40.00%
6,2026-03-10 00:00:00,2500,10,2024-02-26 00:00:00,2026-03-09 00:00:00,50.00%
7,2026-03-11 00:00:00,2500,6,2024-02-27 00:00:00,2026-03-10 00:00:00,50.00%
8,2026-03-12 00:00:00,2500,8,2024-02-28 00:00:00,2026-03-11 00:00:00,37.50%
9,2026-03-13 00:00:00,2500,8,2024-02-29 00:00:00,2026-03-12 00:00:00,37.50%


Logistic + sigmoid thresholded walk-forward accuracy


,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,0.500000,263,100.0%,48.29,53.89
1,0.520000,190,72.2%,47.37,55.03
2,0.550000,73,27.8%,39.73,57.17
3,0.580000,21,8.0%,47.62,59.66
4,0.600000,9,3.4%,66.67,60.92
5,0.650000,0,0.0%,nan,nan


XGBoost Optuna + sigmoid mean day-by-day Accuracy: 54.49%


,date,train_n_games,test_n_games,train_start_date,train_end_date,Accuracy
0,2026-03-04 00:00:00,2500,5,2024-02-13 00:00:00,2026-03-03 00:00:00,60.00%
1,2026-03-05 00:00:00,2500,9,2024-02-14 00:00:00,2026-03-04 00:00:00,66.67%
2,2026-03-06 00:00:00,2500,7,2024-02-14 00:00:00,2026-03-05 00:00:00,28.57%
3,2026-03-07 00:00:00,2500,5,2024-02-23 00:00:00,2026-03-06 00:00:00,80.00%
4,2026-03-08 00:00:00,2500,8,2024-02-24 00:00:00,2026-03-07 00:00:00,50.00%
5,2026-03-09 00:00:00,2500,5,2024-02-25 00:00:00,2026-03-08 00:00:00,60.00%
6,2026-03-10 00:00:00,2500,10,2024-02-26 00:00:00,2026-03-09 00:00:00,40.00%
7,2026-03-11 00:00:00,2500,6,2024-02-27 00:00:00,2026-03-10 00:00:00,83.33%
8,2026-03-12 00:00:00,2500,8,2024-02-28 00:00:00,2026-03-11 00:00:00,50.00%
9,2026-03-13 00:00:00,2500,8,2024-02-29 00:00:00,2026-03-12 00:00:00,62.50%


XGBoost Optuna + sigmoid thresholded walk-forward accuracy


,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,0.500000,263,100.0%,53.23,56.82
1,0.520000,218,82.9%,55.50,58.05
2,0.550000,142,54.0%,56.34,60.47
3,0.580000,85,32.3%,56.47,63.03
4,0.600000,67,25.5%,50.75,64.22
5,0.650000,21,8.0%,42.86,68.82


## Final Comparison Tables


In [30]:
baseline_cv_for_compare = baseline_cv_summary.copy()
baseline_cv_for_compare['validation_brier_score'] = np.nan
baseline_cv_for_compare['validation_log_loss'] = np.nan
baseline_cv_for_compare['validation_ece_10_pct'] = np.nan
baseline_cv_for_compare['validation_n_games'] = np.nan
baseline_cv_for_compare['n_folds'] = np.nan

baseline_final_for_compare = baseline_final_summary.copy()
baseline_final_for_compare['brier_score'] = np.nan
baseline_final_for_compare['log_loss'] = np.nan
baseline_final_for_compare['mean_predicted_over_pct'] = np.nan
baseline_final_for_compare['mean_confidence_pct'] = np.nan
baseline_final_for_compare['ece_10_pct'] = np.nan

comparison_cv_all = pd.concat(
    [baseline_cv_for_compare, calibration_cv_summary],
    ignore_index=True,
)
comparison_final_holdout_all = pd.concat(
    [baseline_final_for_compare, calibration_holdout_summary],
    ignore_index=True,
)

print('Development CV comparison')
display(comparison_cv_all.round(4))

print('Final holdout comparison')
display(comparison_final_holdout_all.round(4))

print('Threshold comparison on development CV predictions')
display(
    calibration_cv_threshold_summary.style.format(
        {
            'pct_of_test': '{:.1%}',
            'bet_accuracy_pct': '{:.2f}',
            'mean_confidence_pct': '{:.2f}',
        }
    )
)

print('Threshold comparison on final holdout predictions')
display(
    calibration_holdout_threshold_summary.style.format(
        {
            'pct_of_test': '{:.1%}',
            'bet_accuracy_pct': '{:.2f}',
            'mean_confidence_pct': '{:.2f}',
        }
    )
)

calibration_diagnostics_cv = calibration_cv_summary[
    [
        'model',
        'validation_brier_score',
        'validation_log_loss',
        'validation_ece_10_pct',
    ]
].copy()
calibration_diagnostics_holdout = calibration_holdout_summary[
    [
        'model',
        'brier_score',
        'log_loss',
        'ece_10_pct',
    ]
].copy()

print('Calibration diagnostics on development CV predictions')
display(calibration_diagnostics_cv.round(4))

print('Calibration diagnostics on final holdout predictions')
display(calibration_diagnostics_holdout.round(4))


Development CV comparison


,model,validation_accuracy_pct,validation_balanced_accuracy_pct,validation_brier_score,validation_log_loss,validation_ece_10_pct,validation_n_games,n_folds
0,BASE_AVG_ALL_6_ERR,54.7393,52.2378,NaN,NaN,NaN,NaN,NaN
1,BASE_MAJORITY_TOTAL_ONLY_ERR,55.4670,53.7951,NaN,NaN,NaN,NaN,NaN
2,BASE_MAJORITY_LINE_ERROR_ONLY_ERR,53.6255,51.3590,NaN,NaN,NaN,NaN,NaN
3,BASE_MAJORITY_ALL_6_TIE_WITH_PRED_LINE_ERROR_F...,55.9866,53.2463,NaN,NaN,NaN,NaN,NaN
4,XGBoost Optuna raw,55.2192,54.4926,0.2473,0.6877,14.0012,461.0000,14.0000
5,XGBoost fixed raw,54.2199,51.0283,0.2485,0.6901,9.8771,461.0000,14.0000
6,Logistic + sigmoid,49.2195,48.0732,0.2514,0.6964,12.8766,461.0000,14.0000
7,XGBoost fixed + sigmoid,51.5484,50.7094,0.2521,0.6975,10.8160,461.0000,14.0000
8,XGBoost Optuna + sigmoid,50.3683,48.7097,0.2540,0.7015,13.4599,461.0000,14.0000
9,Logistic + isotonic,51.0473,51.3555,0.2563,0.7877,12.3722,461.0000,14.0000


Final holdout comparison


,model,n_games,accuracy_pct,balanced_accuracy_pct,brier_score,log_loss,mean_predicted_over_pct,mean_confidence_pct,ece_10_pct
0,BASE_AVG_ALL_6_ERR,263,50.5703,50.8417,NaN,NaN,NaN,NaN,NaN
1,BASE_MAJORITY_TOTAL_ONLY_ERR,263,51.3308,51.6025,NaN,NaN,NaN,NaN,NaN
2,BASE_MAJORITY_LINE_ERROR_ONLY_ERR,263,53.2319,53.4103,NaN,NaN,NaN,NaN,NaN
3,BASE_MAJORITY_ALL_6_TIE_WITH_PRED_LINE_ERROR_F...,263,52.8517,53.0661,NaN,NaN,NaN,NaN,NaN
4,Logistic raw,263,52.8517,52.6322,0.3042,0.8815,56.8086,73.7095,20.8578
5,Logistic + sigmoid,263,49.0494,50.0000,0.2527,0.6987,45.7566,54.2434,5.1940
6,Logistic + isotonic,263,49.0494,50.0000,0.2528,0.6988,45.6001,54.3999,5.3504
7,XGBoost fixed raw,263,50.1901,50.9459,0.2500,0.6931,48.6108,51.5143,2.3397
8,XGBoost fixed + sigmoid,263,50.9506,51.7644,0.2518,0.6968,45.7350,54.4511,5.2156
9,XGBoost fixed + isotonic,263,50.5703,51.1888,0.2516,0.6964,46.2194,54.6769,4.7312


Threshold comparison on development CV predictions


...

,model,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,Logistic raw,0.500000,263,100.0%,52.85,73.71
1,Logistic raw,0.520000,252,95.8%,53.57,74.70
2,Logistic raw,0.550000,237,90.1%,53.59,76.05
3,Logistic raw,0.580000,221,84.0%,54.75,77.47
4,Logistic raw,0.600000,210,79.8%,54.29,78.46
5,Logistic raw,0.650000,185,70.3%,54.59,80.63
6,Logistic + sigmoid,0.500000,263,100.0%,49.05,54.24
7,Logistic + sigmoid,0.520000,263,100.0%,49.05,54.24
8,Logistic + sigmoid,0.550000,0,0.0%,nan,nan
9,Logistic + sigmoid,0.580000,0,0.0%,nan,nan


Calibration diagnostics on development CV predictions


,model,validation_brier_score,validation_log_loss,validation_ece_10_pct
0,XGBoost Optuna raw,0.2473,0.6877,14.0012
1,XGBoost fixed raw,0.2485,0.6901,9.8771
2,Logistic + sigmoid,0.2514,0.6964,12.8766
3,XGBoost fixed + sigmoid,0.2521,0.6975,10.8160
4,XGBoost Optuna + sigmoid,0.2540,0.7015,13.4599
5,Logistic + isotonic,0.2563,0.7877,12.3722
6,XGBoost Optuna + isotonic,0.2507,0.8396,12.4157
7,XGBoost fixed + isotonic,0.2582,1.1785,13.4465
8,Logistic raw,0.3821,1.4041,37.9658


Calibration diagnostics on final holdout predictions


,model,brier_score,log_loss,ece_10_pct
0,Logistic raw,0.3042,0.8815,20.8578
1,Logistic + sigmoid,0.2527,0.6987,5.1940
2,Logistic + isotonic,0.2528,0.6988,5.3504
3,XGBoost fixed raw,0.2500,0.6931,2.3397
4,XGBoost fixed + sigmoid,0.2518,0.6968,5.2156
5,XGBoost fixed + isotonic,0.2516,0.6964,4.7312
6,XGBoost Optuna raw,0.2562,0.7064,7.7599
7,XGBoost Optuna + sigmoid,0.2557,0.7049,8.7780
8,XGBoost Optuna + isotonic,0.2671,0.7449,11.5960
